# Ella2027 — LTX Kaggle Worker (Run All)
نسخة تشغيل واحدة: شغّل Run All واترك جلسة Kaggle مفتوحة. يتم تثبيت المكتبات، فحص الاتصال، تحميل نموذج LTX، ثم استقبال طلبات Ella2027 تلقائياً.

**مرة واحدة فقط قبل التشغيل:** فعّل GPU وInternet من إعدادات الدفتر، واحفظ Kaggle Secret باسم `ELLA_KAGGLE_WORKER_TOKEN` بنفس قيمة متغير Render. لا تضع المفتاح داخل الكود.

ملاحظة: أول تشغيل يحتاج تنزيل النموذج الكبير. جودة الإخراج محسّنة بإعداد توليد أعلى ثم تكبير إخراج 2x وتحسين حدة خفيف؛ التكبير لا يعيد تفاصيل غير موجودة في المصدر. مدة الجلسة وحدود GPU في Kaggle ما زالت تنطبق.

In [ ]:
import os, sys, time, subprocess, shutil, gc
print('=== Ella2027 setup ===')
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg requests
import torch, requests
from pathlib import Path
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'فعّل GPU من Notebook Settings ثم أعد Run All.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
BASE_URL = 'https://ella2027.onrender.com'
TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WORKER_TOKEN')
assert TOKEN, 'Kaggle Secret ELLA_KAGGLE_WORKER_TOKEN غير موجود.'
health = requests.get(BASE_URL + '/api/kaggle-worker/health', headers={'Authorization': 'Bearer ' + TOKEN}, timeout=60)
print('Render health:', health.status_code, health.text)
health.raise_for_status()
assert shutil.which('ffmpeg'), 'ffmpeg غير متوفر في جلسة Kaggle.'
print('Setup and connection OK.')

In [ ]:
from diffusers import LTXPipeline
from diffusers.utils import export_to_video
MODEL_ID = 'Lightricks/LTX-Video'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Loading {MODEL_ID} with {dtype}; first run downloads model weights...')
pipe = LTXPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_tiling()
    print('VAE tiling enabled.')
except Exception as e:
    print('VAE tiling not available; continuing:', str(e)[:180])
gc.collect()
torch.cuda.empty_cache()
print('LTX pipeline ready.')

In [ ]:
!mkdir -p /kaggle/working/ella_worker
!wget -q -O /kaggle/working/ella_worker/kaggle_worker.py https://raw.githubusercontent.com/WaelNameh84/Ella2027/main/services/kaggle-ltx-worker/kaggle_worker.py
sys.path.insert(0, '/kaggle/working/ella_worker')
from kaggle_worker import run_kaggle_ltx_worker

def generate_with_ltx(job):
    prompt = str(job.get('prompt') or 'Cinematic realistic scene')
    seconds = min(max(int(job.get('durationSeconds') or 5), 1), 5)
    fps = 8
    frames = seconds * fps + 1  # LTX temporal frame alignment
    # Quality-balanced settings for the standard LTX-Video checkpoint on a T4.
    width, height = 768, 448
    steps = 24
    negative_prompt = 'blurry, low detail, low resolution, smeared textures, distorted faces, extra limbs, flicker, jitter, unstable motion, artifacts, watermark, text'
    print(f'Generating {seconds}s at {width}x{height}, {frames} frames, {steps} steps')
    result = pipe(prompt=prompt, negative_prompt=negative_prompt,
                  width=width, height=height, num_frames=frames,
                  num_inference_steps=steps, guidance_scale=5.0)
    raw = Path('/kaggle/working') / f'ella_{job["id"]}_raw.mp4'
    out = Path('/kaggle/working') / f'ella_{job["id"]}.mp4'
    export_to_video(result.frames[0], str(raw), fps=fps)
    # 2x Lanczos output sizing + restrained sharpening; source is now 768x448.
    cmd = ['ffmpeg','-y','-i',str(raw),'-vf',
           'scale=1536:896:flags=lanczos,unsharp=5:5:0.45:3:3:0.15',
           '-c:v','libx264','-preset','veryfast','-crf','18','-pix_fmt','yuv420p','-an',str(out)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError('FFmpeg quality pass failed: ' + (proc.stderr or '')[-1200:])
    if not out.is_file() or out.stat().st_size == 0:
        raise RuntimeError('LTX did not produce a valid MP4')
    try: raw.unlink()
    except Exception: pass
    print('Finished:', out.name, f'{out.stat().st_size/1024/1024:.1f} MiB')
    return out

print('Ella2027 worker ready. Waiting for video jobs...')
run_kaggle_ltx_worker(BASE_URL, TOKEN, generate_with_ltx, idle_seconds=10)
